In [40]:
import json
import random
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode
from zoneinfo import ZoneInfo

import pandas as pd
import requests


# Network Reverse Engineering
* using daangn's internal API

In [ ]:
BASE_URL = "https://www.daangn.com/kr/buy-sell/s/"
DATA_ROUTE = "routes/kr.buy-sell.s"

keyword = "아동복"

def build_in_param(region_name: str, region_id: int) -> str:
    # 예: 삼성동-6034
    return f"{region_name}-{region_id}"


def build_url(in_param: str, search: str) -> str:
    qs = {"in": in_param, "search": search, "_data": DATA_ROUTE}
    return BASE_URL + "?" + urlencode(qs)


def safe_filename(s: str) -> str:
    return "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in s)


def fetch_json(url: str, headers: dict, timeout: int = 25, max_retries: int = 5) -> dict:
    backoff = 1.7
    for attempt in range(1, max_retries + 1):
        r = requests.get(url, headers=headers, timeout=timeout)

        if r.status_code == 200:
            ct = (r.headers.get("Content-Type") or "").lower()
            if "json" not in ct:
                raise RuntimeError(f"JSON이 아닌 응답(Content-Type={ct}). 차단/리다이렉트 가능성")
            return r.json()

        if r.status_code in (403, 429):
            time.sleep((backoff ** attempt) + random.uniform(0.3, 1.0))
            continue

        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")

    raise RuntimeError("재시도 초과 (차단/인증/레이트리밋 가능)")


def extract_regions(regions_json_path: str, include_children: bool = False) -> list[dict]:
    with open(regions_json_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    opts = (payload.get("regionFilterOptions") or {})
    regions = []

    cur = opts.get("region")
    if isinstance(cur, dict) and cur.get("id") and cur.get("name"):
        regions.append(cur)

    sibs = opts.get("siblingRegions") or []
    if isinstance(sibs, list):
        regions.extend([r for r in sibs if isinstance(r, dict) and r.get("id") and r.get("name")])

    if include_children:
        kids = opts.get("childrenRegions") or []
        if isinstance(kids, list):
            regions.extend([r for r in kids if isinstance(r, dict) and r.get("id") and r.get("name")])

    # 중복 제거 (id 기준)
    uniq = {}
    for r in regions:
        uniq[int(r["id"])] = r
    return list(uniq.values())


def crawl_all_regions_to_one_file(
    regions_json_path: str,
    search: str,
    out_dir: str = ".",
    tz: str = "Asia/Seoul",
    cookie: str | None = None,
    include_children: bool = False,
    sleep_min: float = 0.8,
    sleep_max: float = 1.8,
) -> str:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    regions = extract_regions(regions_json_path, include_children=include_children)

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "application/json,text/plain,*/*",
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "Referer": "https://www.daangn.com/",
    }
    if cookie:
        headers["Cookie"] = cookie  # 막히면 DevTools에서 Cookie 복사해 넣기

    crawled_at = datetime.now(ZoneInfo(tz)).strftime("%Y-%m-%d %H:%M:%S")  # 보기 쉬운 KST

    all_articles = []
    errors = []

    for idx, r in enumerate(regions, start=1):
        region_id = int(r["id"])
        region_name = str(r["name"])
        in_param = build_in_param(region_name, region_id)

        url = build_url(in_param=in_param, search=search)

        try:
            payload = fetch_json(url, headers=headers)
            articles = ((payload.get("allPage") or {}).get("fleamarketArticles")) or []

            # 기사(게시물) 단위로 지역/수집시간 메타 붙여서 누적
            for a in articles:
                a["_meta"] = {
                    "crawledAt": crawled_at,
                    "region_id": region_id,
                    "region_name": region_name,
                    "region_in": in_param,
                    # 원하면 상위 지역도 같이 저장 가능
                    "name1Id": r.get("name1Id"),
                    "name2": r.get("name2"),
                    "name3": r.get("name3"),
                    "depth": r.get("depth"),
                }
                all_articles.append(a)

            print(f"[{idx}/{len(regions)}] ok  region={region_name}({region_id})  articles={len(articles)}")

        except Exception as e:
            msg = f"region={region_name}({region_id}) err={repr(e)}"
            errors.append(msg)
            print(f"[{idx}/{len(regions)}] FAIL  {msg}")

        time.sleep(random.uniform(sleep_min, sleep_max))

    out_path = out / f"search_{safe_filename(search)}.json"
    bundle = {
        "search": search,
        "crawledAt": crawled_at,
        "regions_count": len(regions),
        "articles_count": len(all_articles),
        "errors": errors,
        "articles": all_articles,
    }

    with open(out_path, "w", encoding="utf-8") as wf:
        json.dump(bundle, wf, ensure_ascii=False, indent=2)

    print(f"\nSaved -> {out_path} (articles={len(all_articles)}, errors={len(errors)})")
    return str(out_path)


crawl_all_regions_to_one_file(
    regions_json_path="../data/json/test-regions.json",
    search=keyword,
    out_dir="../data/json",
    cookie=None,
    include_children=False,  # 필요하면 True
)

# json-to-DataFrame

In [ ]:
def bundle_json_to_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    rows = []
    for a in bundle.get("articles", []):
        user = a.get("user") or {}
        region = a.get("region") or {}
        meta = a.get("_meta") or {}

        rows.append({
            'id': a.get('id').split('-')[-1][:-1],
            "href": a.get("href"),
            "price": a.get("price"),
            "title": a.get("title"),
            "status": a.get("status"),
            "content": a.get("content"),
            "createdAt": a.get("createdAt"),
            "boostedAt": a.get("boostedAt"),
            "user_dbId": user.get("dbId"),
            "user_nickname": user.get("nickname"),
            "region_name_from_article": region.get("name"),

            # ✅ 관리용(요청 지역 정보)
            "region_id": meta.get("region_id"),
            "region_name": meta.get("region_name"),
            "region_in": meta.get("region_in"),
            "crawledAt": meta.get("crawledAt"),
        })

    df = pd.DataFrame(rows)
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["createdAt"] = pd.to_datetime(df["createdAt"], errors="coerce")
    df["boostedAt"] = pd.to_datetime(df["boostedAt"], errors="coerce")
    df["crawledAt"] = pd.to_datetime(df["crawledAt"], errors="coerce").dt.tz_localize("Asia/Seoul")
    for col in ["createdAt", "boostedAt", "crawledAt"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df[col] = df[col].dt.tz_convert("Asia/Seoul")
    return df


# 사용 예:
df = bundle_json_to_df(f"../data/json/search_{keyword}.json")
df.head(1)

,id,href,price,title,status,content,createdAt,boostedAt,user_dbId,user_nickname,region_name_from_article,region_id,region_name,region_in,crawledAt
0,8wqyok7b9sf6,https://www.daangn.com/kr/buy-sell/%ED%8F%B4%E...,71000.0,폴로 랄프로렌 에스테트립 하프 집업팝니다.,Ongoing,구입 후 보관만 한 새상품급 에스테이트립 집업입니다.\n백화점 구입 정품입니다.\n...,2026-02-25 20:51:02.197000+09:00,2026-03-03 17:12:18.814000+09:00,11016644,당근당근당근,삼성동,6034,삼성동,삼성동-6034,2026-03-03 17:19:55+09:00


In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8970 entries, 0 to 8969
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype                     
---  ------                    --------------  -----                     
 0   id                        8970 non-null   str                       
 1   href                      8970 non-null   str                       
 2   price                     8963 non-null   float64                   
 3   title                     8970 non-null   str                       
 4   status                    8970 non-null   str                       
 5   content                   8970 non-null   str                       
 6   createdAt                 8970 non-null   datetime64[us, Asia/Seoul]
 7   boostedAt                 8970 non-null   datetime64[us, Asia/Seoul]
 8   user_dbId                 8970 non-null   str                       
 9   user_nickname             8970 non-null   str                       
 10  region_name